# Dataset folder structure
```
data/
│
├── images/                              # PNG files
│   ├── 0.png                           
│   │   .
│   │   .
│   │   .
│   └── 1.png                          
│
└── labels/                                # txt files
	├── 0.txt
	.
	.
	.
	└── 1.txt
```

In [ ]:
from pathlib import Path

MODEL='ETLTC F'

EPOCHS=1

LR=5e-6     #1e-4

DATA_PATH = Path('data/kkanji2')

USE_CHECKPOINT = True

DATASET_PATH = Path('preprocessed_data/kkanji2')

K49_PATH = Path('preprocessed_data/K49')

DATA_PATH.mkdir(parents=True, exist_ok=True)

DATASET_PATH.mkdir(parents=True, exist_ok=True)

PREPROCESS_DATA = False

AUGMENT_DATA = False

In [ ]:
from dtrocr.config import DTrOCRConfig

config = DTrOCRConfig(
    # attn_implementation='flash_attention_2'
)

In [ ]:
from util.model import get_model

model = get_model(config, MODEL)

In [ ]:
from dtrocr.processor import DTrOCRProcessor
model.eval()
model.to('cpu')
test_processor = DTrOCRProcessor(config=config, add_bos_token=True, add_eos_token=True)
test_processor.tokeniser


In [ ]:
from util.data_processing import get_words_list
import json

with open(str(DATASET_PATH / 'test_labels.json'), 'r') as fp:
    test_dict = json.load(fp)

    
test_words_files = list(test_dict.items())
test_words = get_words_list(test_words_files)

In [ ]:
char_list = ["髭", "掟", "鎚", "祚", "荼"]

In [ ]:
for word in test_words:
    if word.transcription.strip() in char_list:
        print(f"{word.transcription.strip()}: {word.file_path}")

In [ ]:
import pickle

with open(f'class_accuracy/class_frequency_map.pkl', 'rb') as f:
    class_frequency_map = pickle.load(f)
for word, frequency in class_frequency_map.items():
    if (word in char_list):
        print(frequency)

In [ ]:
from PIL import Image
import torch

image_file = test_word_record.file_path
image = Image.open(image_file).convert('RGB')

inputs = test_processor(
    images=image, 
    texts=test_word_record.transcription,
    padding='max_length',
    return_tensors='pt',
    return_labels=True
)
model_output, acc = model.generate(
    inputs, 
    test_processor,
    num_beams = 1
)

predicted_text = test_processor.tokeniser.decode(model_output[0], skip_special_tokens=True)
mask = inputs.label_attention_mask[..., 1:].reshape(-1)
shift_labels = inputs.labels[..., 1:].contiguous()
print(inputs.labels)
print(shift_labels)
print('Ответ: ', test_word_record.transcription, ' Вывод модели: ', predicted_text, ' Acc: ', acc)
print(model_output)

In [ ]:
inputs = {'pixel_values': inputs.pixel_values[0],
            'input_ids': inputs.input_ids[0],
            'input_attention_mask': inputs.input_attention_mask[0],
            'label_attention_mask': inputs.label_attention_mask[0],
            'labels': inputs.labels[0]}
outputs = model(**inputs)
accuracy = outputs.accuracy.item()
print(accuracy)

In [ ]:
print(test_processor.tokeniser.decode(377))
print(test_processor.tokeniser.eos_token, test_processor.tokeniser.pad_token, test_processor.tokeniser.bos_token)
print(test_processor.tokeniser.encode(test_processor.tokeniser.bos_token))